In [1]:
import os
import sys
import scipp as sc
import mcstastox as mx

parent = os.path.dirname(os.getcwd())
sys.path.append(parent)


van_path = parent + "/runs/LET_vanad_large"

with mx.Read(van_path) as mcstas_van_data:
    scipp_van_data_group = mcstas_van_data.export_scipp(
        source_name="SourceMantid",
        sample_name="iso_samp",
    )

v_events_binned = scipp_van_data_group["events"]
display(v_events_binned)
# Group by pixel_id to fill in the pixels that recorded no events
det_positions = scipp_van_data_group["positions"]
sample_position = v_events_binned.coords["sample_position"]
v_events = v_events_binned.group(sc.arange(det_positions.dim, 0, det_positions.size))
display(v_events)

<scipp.DataArray>
Dimensions: Sizes[pixel_id:30056, ]
Coordinates:
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 30054, 30055]
* position                  vector3              [m]  (pixel_id)  [(0.323627, -1.98529, 28.485), (0.360764, -1.98529, 28.4814), ..., (2.29234, 1.98529, 22.3552), (2.26402, 1.98529, 22.3309)]
* sample_position           vector3              [m]  ()  (0, 0, 25)
* source_position           vector3              [m]  ()  (0, 0, 0)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 37787068),
          data=float64[counts],
          coords={'t':float64[s]})

<scipp.DataArray>
Dimensions: Sizes[pixel_id:30056, ]
Coordinates:
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 30054, 30055]
* sample_position           vector3              [m]  ()  (0, 0, 25)
* source_position           vector3              [m]  ()  (0, 0, 0)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 37787068),
          data=float64[counts],
          coords={'t':float64[s]})

In [2]:
# solid angles
d_omega = v_events.hist().data
d_omega /= d_omega.sum()
d_omega

<scipp.Variable> (pixel_id: 30056)    float64  [dimensionless]  [2.73278e-05, 2.48994e-05, ..., 2.67895e-05, 2.86404e-05]

In [3]:
# LET banana detectors
# Horizontal +5 to 140 degs, sample to detector distance is R = 3.5 m, 221 bins
# width per pixel is w = (140-5)/180*pi*R/221 = (135/180)* 0.04975 m =  0.037315
# Vertical height is H = 4 m, 136 bins
# height per pixel is h = H/136 = 0.0294 m
# solid angle Omega = 4*arctan(w*h/(2d*sqrt(4d^2+w^2+h^2)))
# d = sqrt(x^2+y^2+z^2)
import numpy as np

h = sc.norm(det_positions[221] - det_positions[0])
w = sc.norm(det_positions[1] - det_positions[0])
r = det_positions - sample_position
d = sc.norm(r)
ratio = w * h / 4 / d / sc.sqrt(d**2 + w**2 / 4 + h**2 / 4)

d_omega_calc = sc.array(dims=["pixel_id"], values=4 * np.arctan(ratio.values))

x = r.fields.x
y = r.fields.y
z = r.fields.z
rho = sc.sqrt(x**2 + z**2)
r_plus = sc.sqrt(rho**2 + w**2 / 4 + (y + h / 2) ** 2)
r_minus = sc.sqrt(rho**2 + w**2 / 4 + (y - h / 2) ** 2)
ratio_plus = w / 2 * (y + h / 2) / rho / r_plus
ratio_minus = w / 2 * (y - h / 2) / rho / r_minus
d_omega_calc_vert = sc.array(
    dims=["pixel_id"],
    values=2 * (np.arctan(ratio_plus.values) - np.arctan(ratio_minus.values)),
)
# norm_factors.coords["d_omega_calc"] = d_omega_calc

In [4]:
%matplotlib widget
import plopp as pp

pp.plot(
    {
        "Van": d_omega,
        "Calc normal (x,y,z)": d_omega_calc / d_omega_calc.sum(),
        "Calc normal (x,0,z)": d_omega_calc_vert / d_omega_calc_vert.sum(),
    },
    ylabel="fractional counts",
    title="Vanadium counts/pixel over total counts",
    linestyle="-",
    markersize=2,
    ymax=1e-4,
    ymin=0.0,
    grid="True",
)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [5]:
import numpy as np

nx, ny = 221, 136
data = np.reshape(d_omega.values, (ny, nx))

pp.plot(data)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…